# Load Pre-trained Embedding Model

In [2]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, concatenate_datasets
import torch

model_id = "Snowflake/snowflake-arctic-embed-m"  # Use a reasonably good model here

model_retrieval = SentenceTransformer(
    model_id, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Load and Combine Multiple Datasets

In [3]:
from datasets import load_dataset, concatenate_datasets

# Load each split separately
easy_para = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_batch10", "easy_triplets_paraphrase")["train"]
hard_para = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_batch10", "hard_triplets_paraphrase")["train"]
hard_sem = load_dataset("frankwong2001/ssf-dataset_Full_synthetic_batch10", "hard_triplets_semantic")["train"]

# Concatenate all splits into one dataset
dataset = concatenate_datasets([easy_para, hard_para, hard_sem])

# Clean Dataset Structure

In [4]:
dataset = dataset.select_columns(['anchor', 'positive', 'negative'])
dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 5655
})

# Define Quality Assessment Functions
### These functions compute semantic similarity scores between different text pairs in our triplets:
- get_embeddings(): Converts text to vector representations using our embedding model
- get_similarities(): Computes cosine similarity between embedding vectors
- format_data_retriever(): Processes batches of triplets to add similarity scores

### The similarity scores help us identify high-quality triplets where:
- Anchor-positive pairs have high similarity (good matches)
- Anchor-negative pairs have moderate similarity (hard negatives)
- Positive-negative pairs are sufficiently different

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def get_embeddings(texts):
    vectors = model_retrieval.encode(texts)
    return [vector.tolist() for vector in vectors]


def get_similarities(vector_batch_a, vector_batch_b):
    similarities = []
    for vector_a, vector_b in zip(vector_batch_a, vector_batch_b):
        similarity = cosine_similarity([vector_a], [vector_b])[0][0]
        similarities.append(similarity)
    return similarities

def format_data_retriever(batch):# -&gt; Any:
    batch["anchor-vector"] = get_embeddings(batch["anchor"])
    batch["positive-vector"] = get_embeddings(batch["positive"])
    batch["negative-vector"] = get_embeddings(batch["negative"])    
    batch["similarity-positive-negative"] = get_similarities(batch["positive-vector"], batch["negative-vector"])
    batch["similarity-anchor-positive"] = get_similarities(batch["anchor-vector"], batch["positive-vector"])
    batch["similarity-anchor-negative"] = get_similarities(batch["anchor-vector"], batch["negative-vector"])
    return batch

# Compute Similarity Scores

In [6]:
dataset = dataset.map(format_data_retriever, batched=True, batch_size=250)

Map:   0%|          | 0/5655 [00:00<?, ? examples/s]

# Inspect the Enriched Dataset

In [7]:
dataset.to_pandas()

,anchor,positive,negative,anchor-vector,positive-vector,negative-vector,similarity-positive-negative,similarity-anchor-positive,similarity-anchor-negative
0,The Audit Associate/Audit Assistant Associate ...,The Audit Associate/Audit Assistant Associate ...,The Marketing Coordinator develops promotional...,"[0.06092929095029831, 0.09926585108041763, 0.0...","[0.05978068336844444, 0.09333156049251556, 0.0...","[0.04162754490971565, 0.03798741474747658, -0....",0.512734,0.951419,0.493079
1,The Audit Senior Manager/Audit Manager manages...,The Audit Senior Manager/Audit Manager is resp...,The Marketing Coordinator organizes promotiona...,"[0.057781320065259933, 0.07172407954931259, -0...","[0.04016847908496857, 0.07711590081453323, -0....","[0.03980385884642601, 0.033586133271455765, -0...",0.518041,0.958153,0.525124
2,The Audit Partner/Audit Director is a transfor...,The Audit Partner/Audit Director acts as a tra...,The Software Developer designs and implements ...,"[0.03437991812825203, 0.07796177268028259, -0....","[0.03421350196003914, 0.09226571023464203, -0....","[-0.009419537149369717, 0.08730371296405792, -...",0.501475,0.974504,0.468217
3,The Audit Senior is expected to team lead vari...,The Audit Senior is responsible for leading va...,The Chef de Partie oversees the preparation an...,"[0.02032621204853058, 0.06756345182657242, -0....","[0.008157119154930115, 0.07263972610235214, -0...","[-0.030600620433688164, 0.11190570145845413, -...",0.606131,0.948573,0.601098
4,The Business Valuation Associate/Business Valu...,The Business Valuation Associate/Business Valu...,The Software Developer designs and implements ...,"[0.032666150480508804, 0.05040699988603592, -0...","[0.027943044900894165, 0.054589368402957916, -...","[-0.013902525417506695, 0.09286864101886749, -...",0.558173,0.978670,0.559743
...,...,...,...,...,...,...,...,...,...
5650,The WSH Manager is responsible for reviewing W...,The WSH Manager is tasked with reviewing workp...,The WSH Coordinator is accountable for assessi...,"[-0.01395051833242178, 0.08462846279144287, -0...","[-0.018405813723802567, 0.07409617304801941, -...","[0.012552754022181034, 0.07444283366203308, -0...",0.884464,0.973614,0.866714
5651,The WSH Officer is responsible for developing ...,The WSH Officer is tasked with creating and ov...,The WSH Coordinator is in charge of implementi...,"[-0.0010848931269720197, 0.05022454261779785, ...","[-0.013112182728946209, 0.052766185253858566, ...","[0.01069618295878172, 0.0657278448343277, -0.0...",0.865313,0.938675,0.829309
5652,The Workplace Safety and Health (WSH) Supervis...,The Workplace Safety and Health (WSH) Supervis...,The Workplace Health and Safety (WHS) Coordina...,"[0.020672142505645752, 0.08999121934175491, -0...","[0.011982506141066551, 0.09165038168430328, -0...","[0.05979115515947342, 0.06661400198936462, -0....",0.827011,0.973439,0.827766
5653,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Workplace Safety and Health (WSH) Aud...,The Lead Quality Assurance (QA) Officer is in ...,"[-0.003643269184976816, 0.043617475777864456, ...","[-0.013482298702001572, 0.05899694934487343, -...","[-0.01015164889395237, 0.04505638778209686, -0...",0.680992,0.966771,0.658834


# Apply Quality Filtering Criteria
### Filtering Rules:
1. Anchor-Positive similarity > 0.6: Ensures positive examples are semantically related to anchors
2. Anchor-Negative similarity 0.2-0.6: Creates "hard negatives" that are somewhat related but not too similar
3. Positive-Negative similarity < 0.7: Ensures positive and negative examples are sufficiently distinct

In [ ]:
def filter_with_hard_negatives(example):
    anchor_pos = example["similarity-anchor-positive"]
    anchor_neg = example["similarity-anchor-negative"] 
    pos_neg = example["similarity-positive-negative"]
    
    # return (
    #     anchor_pos > 0.6 and  # Good positive
    #     0.2 < anchor_neg < 0.6 and  # Hard negative range
    #     pos_neg < 0.7  # Positive and negative are distinct
    # )

    return (
        anchor_pos > 0.78 and  # Good positive
        0.2 < anchor_neg < 0.78 and  # Hard negative range
        pos_neg < 0.7  # Positive and negative are distinct (ask dickson why need this)
    )


cleaned_dataset = dataset.filter(filter_with_hard_negatives)

Filter:   0%|          | 0/5655 [00:00<?, ? examples/s]

# Clean Filtered Dataset
### Remove the embedding vectors and similarity scores to keep only the essential text data. This reduces storage requirements while preserving the high-quality triplets identified by our filtering process.

In [9]:
cleaned_dataset = cleaned_dataset.select_columns(['anchor', 'positive', 'negative'])
cleaned_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1407
})

# Split into Training and Validation Sets

In [10]:
train_size = int(0.8 * len(cleaned_dataset))
valid_size = len(cleaned_dataset) - train_size

train_dataset = cleaned_dataset.select(range(train_size))
valid_dataset = cleaned_dataset.select(range(train_size, train_size + valid_size))

# Verify Dataset Splits

In [11]:
train_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 1125
})

In [12]:
valid_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 282
})

In [13]:
from datasets import DatasetDict

ds = DatasetDict({
    "train": train_dataset,
    "valid": valid_dataset
})

ds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1125
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 282
    })
})

# Create Dataset Dictionary

In [14]:
ds.push_to_hub("frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  45%|####5     |  527kB / 1.16MB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########|  240kB /  240kB            

CommitInfo(commit_url='https://huggingface.co/datasets/frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned/commit/7ed5d68d8ca15ee4c388940a248a69b5d8f345b9', commit_message='Upload dataset', commit_description='', oid='7ed5d68d8ca15ee4c388940a248a69b5d8f345b9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned', endpoint='https://huggingface.co', repo_type='dataset', repo_id='frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned'), pr_revision=None, pr_num=None)